# Market Basket Analysis

## Business Problem

Olist wants to understand which product categories are frequently purchased together.

This analysis helps the business create product bundles, improve cross-selling, and recommend related products to customers.

## Business Questions

1. Which product categories are most frequently purchased together?

2. What product combinations have the strongest association?

3. Which products should be bundled to increase average order value?

4. Which product recommendations can improve customer experience?

5. How can product associations support cross-selling campaigns?

In [39]:
# import 
import pandas as pd 
from mlxtend.frequent_patterns import apriori, association_rules
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [40]:
# Load dataset from CSV file into a pandas DataFrame.
#orders_df` tells us which orders were completed
#order_items_df tells us which products were inside each order
#products_df tells us the product category

DATA_DIR = Path("../data")

orders_df = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
products_df=pd.read_csv(DATA_DIR/"olist_products_dataset.csv")
order_items_df=pd.read_csv(DATA_DIR/"olist_order_items_dataset.csv")
product_category_name_translation_df=pd.read_csv(DATA_DIR/"product_category_name_translation.csv")



In [41]:
# Add English category names to the products dataset
products_df = products_df.merge(
    product_category_name_translation_df,
    on="product_category_name",
    how="left"
)

In [43]:
products_df.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [42]:
products_df.isnull().sum()

product_id                         0
product_category_name            610
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english    623
dtype: int64

## Step 3: Filter Delivered Orders

We only keep delivered orders because Market Basket Analysis should be based on completed purchases.

Canceled, unavailable, or pending orders do not represent successful customer buying behavior.

In [44]:
delivered_orders_df=orders_df.loc[orders_df['order_status']=='delivered',['order_id']]

## Step 4: Create Market Basket Dataset

We merge delivered orders with order items and product categories.

This creates one dataset showing which product categories were purchased in each completed order.

In [45]:
# Merge delivered orders with order items and product categories
market_basket_df = (
    delivered_orders_df
    .merge(order_items_df, on="order_id")
    .merge(
        products_df[
            ["product_id", "product_category_name_english"]
        ],
        on="product_id"
    )
)

In [46]:
# Remove products without valid categories
market_basket_df = market_basket_df.dropna(
    subset=["product_category_name_english"]
)

## Step 5: Create Basket Matrix

The Apriori algorithm needs data in a basket format.

Each row represents one order.  
Each column represents one product category.  
The value is:

- `1` = category was purchased in that order
- `0` = category was not purchased in that order


We want one row per order, with every product category shown as a column.

If the category was bought in that order, the value should be 1.
If it was not bought, the value should be 0.

To create this structure, we group by `order_id` and `product_category_name`, then use `.size()` to count how many times each category appears in each order.

After that, `.unstack(fill_value=0)` moves product categories from rows into columns and fills categories that were not purchased with 0.

Finally, we convert all values greater than 0 to 1 because Market Basket Analysis only needs to know whether the category was purchased or not.

In [47]:
# Count category occurrences per order
basket_df = (
    market_basket_df
    .groupby(
        ["order_id", "product_category_name_english"]
    )
    .size()
    .unstack(fill_value=0)
)
# Convert counts to purchased/not purchased
basket_df = (basket_df > 0).astype(int)

## Step 6: Generate Frequent Itemsets

We use the Apriori algorithm to identify product category combinations that occur frequently across orders.

The `min_support` threshold filters out rare combinations and keeps only meaningful purchasing patterns.

In [51]:
# Generate frequent itemsets using the Apriori algorithm
frequent_itemsets_df = apriori(
    basket_df,
    min_support=0.005,
    use_colnames=True
)

# Sort itemsets by support
frequent_itemsets_df = frequent_itemsets_df.sort_values(
    by="support",
    ascending=False
)

# Display the most frequent itemsets
frequent_itemsets_df.head(10)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
2,0.097470,frozenset({bed_bath_table})
12,0.090900,frozenset({health_beauty})
22,0.079157,frozenset({sports_leisure})
4,0.068645,frozenset({computers_accessories})
10,0.066301,frozenset({furniture_decor})
15,0.060372,frozenset({housewares})
26,0.057765,frozenset({watches_gifts})
24,0.043027,frozenset({telephony})
0,0.040052,frozenset({auto})
25,0.039989,frozenset({toys})


## Step 7: Identify Product Combinations

Single-item itemsets show the most popular categories.

To discover cross-selling opportunities, we focus on combinations containing two or more product categories.

In [52]:
# Filter itemsets containing two or more categories
multi_itemsets_df = frequent_itemsets_df[
    frequent_itemsets_df["itemsets"].apply(len) >= 2
]

# Sort by support
multi_itemsets_df = multi_itemsets_df.sort_values(
    by="support",
    ascending=False
)

# Display top combinations
multi_itemsets_df.head(10)

,support,itemsets


In [53]:
#check how many different caterory in every row
items_per_order_df = basket_df.sum(axis=1).value_counts().sort_index()

items_per_order_df

1    94405
2      707
3       15
Name: count, dtype: int64

In [54]:
(basket_df.sum(axis=1) >= 2).mean()

np.float64(0.007589853564182619)

### Initial Finding

Most Olist orders contain products from a single category.

Only approximately 0.76% of orders include items from two or more categories.

This limits the number of frequent cross-category purchasing patterns that can be identified using Market Basket Analysis.

In [55]:
frequent_itemsets_df = apriori(
    basket_df,
    min_support=0.0005,
    use_colnames=True
)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [56]:
multi_itemsets_df = frequent_itemsets_df[
    frequent_itemsets_df["itemsets"].apply(len) >= 2
].sort_values(
    by="support",
    ascending=False
)

multi_itemsets_df.head(10)

,support,itemsets
58,0.000736,"frozenset({bed_bath_table, furniture_decor})"


### Initial Cross-Selling Insight

The combination of `bed_bath_table` and `furniture_decor` appeared together in approximately 0.07% of all delivered orders.

Although support is low, this is expected because most Olist orders contain products from only one category.

The relationship suggests that customers purchasing home-related products may be interested in complementary items, creating opportunities for product bundling and targeted recommendations.

## Business Questions

1. If a customer purchases product category A, how likely are they to purchase product category B?

2. Which product combinations have the strongest relationships?

3. Which categories should be recommended together?

4. Which product bundles can increase average order value?

5. How can purchasing patterns improve cross-selling strategies?

## Why Generate Association Rules?

Frequent itemsets identify categories that appear together frequently.

However, they do not tell us the strength or direction of the relationship.

Association rules help answer questions such as:

- Customers who buy `bed_bath_table` are likely to buy which other categories?
- How strong is the relationship between two categories?
- Is the relationship meaningful or occurring by chance?

These insights support recommendation systems, product bundling, and personalized marketing.

In [58]:
association_rules_df = association_rules(
    frequent_itemsets_df,
    metric="confidence",
    min_threshold=0.001
)

association_rules_df[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].sort_values(
    by="lift",
    ascending=False
).head(10)

,antecedents,consequents,support,confidence,lift
1,frozenset({furniture_decor}),frozenset({bed_bath_table}),0.000736,0.011099,0.113869
0,frozenset({bed_bath_table}),frozenset({furniture_decor}),0.000736,0.007550,0.113869


## Association Rule Metrics

- **Support:** Percentage of orders containing both categories.

- **Confidence:** Probability of purchasing the consequent category when the antecedent category is purchased.

- **Lift:** Measures the strength of the relationship compared to random chance.

Interpretation of lift:

- `Lift > 1` indicates a positive relationship.
- `Lift = 1` indicates no relationship.
- `Lift < 1` indicates a negative relationship.

## Key Findings

Market Basket Analysis revealed limited cross-category purchasing behavior.

Approximately 99% of orders contained products from a single category, resulting in very few meaningful product associations.

The strongest identified combination was `bed_bath_table` and `furniture_decor`; however, the lift score was below 1, indicating that the categories are not purchased together more frequently than expected by chance.

These findings suggest that customers on the Olist platform typically purchase products within a single category per transaction.

# Conclusion

The market basket analysis was conducted to identify product categories that are frequently purchased together and to evaluate opportunities for cross-selling and product bundling.

The analysis revealed that most customer orders contained products from a single category, with approximately 99% of orders showing no cross-category purchasing behavior. While several category combinations appeared frequently together, the overall association strength was weak and most lift scores remained below 1.

The strongest relationship was observed between Bed Bath & Table and Furniture & Decor, which is consistent with customer purchasing behavior for home-related products.

Overall, the results suggest that category-level purchasing behavior is relatively independent, limiting the effectiveness of broad cross-category recommendation strategies.

# Executive Summary

The purpose of this analysis was to determine whether customer purchasing behavior could support cross-selling and product bundling initiatives.

During the analysis, I found that customers typically purchase within a single product category rather than across multiple categories. Although a small number of category combinations appeared together more frequently than others, the overall association strength was weak.

This suggests that category-level recommendation strategies may have limited impact on revenue growth. Instead, future recommendation systems should focus on product-level relationships, customer purchase history, and personalized recommendations.

From a business perspective, the analysis indicates that customer purchasing decisions are driven more by specific product needs than by broad category relationships.

# Management Recommendations

1. Focus future market basket analysis at the product level rather than the category level.
2. Develop personalized recommendation strategies using customer purchase history.
3. Test product bundles within related home categories such as Bed Bath & Table and Furniture & Decor.
4. Combine Market Basket Analysis with RFM segmentation to improve targeting and recommendation relevance.
5. Monitor repeat-purchase behavior to identify sequential buying patterns that may not be visible in same-order analysis.
